# DS201 · Lab 2: Mạng Nơ-ron Tích chập (CNN) trên VinaFood21

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Notebook này dùng để chạy toàn bộ các bài thực hành tuần 2 theo chuẩn module hoá của thư mục `cnn_project`.

**Yêu cầu môi trường Colab:** Chọn **Runtime** -> **Change runtime type** -> **T4 GPU**.

In [ ]:
# 1. Kiểm tra cấu hình GPU
!nvidia-smi

In [ ]:
# 2. Cài đặt các thư viện phụ thuộc
!pip install -q -r requirements.txt

In [ ]:
# 3. Tự động tải và giải nén bộ dữ liệu VinaFood21 từ Google Drive
from data import download_vinafood21
download_vinafood21("VinaFood21")

## Cách 1: Chạy tự động toàn bộ 4 bài thực hành trong 1 lệnh duy nhất
Lệnh dưới đây sẽ chạy lần lượt cả 4 bài, đánh giá trên test set và xuất bảng tổng hợp kết quả.

In [ ]:
!python run_all.py --data_root VinaFood21

## Cách 2: Chạy từng bài riêng biệt (nếu muốn huấn luyện và quan sát từng mô hình)

In [ ]:
# --- BÀI 1: LENET-5 (AvgPool vs MaxPool) ---
!python train.py --model lenet_avg --epochs 20
!python evaluate.py --ckpt outputs/lenet_avg/best.pt

!python train.py --model lenet_max --epochs 20
!python evaluate.py --ckpt outputs/lenet_max/best.pt

In [ ]:
# --- BÀI 2: GOOGLENET (Inception Blocks) ---
!python train.py --model googlenet --epochs 20
!python evaluate.py --ckpt outputs/googlenet/best.pt

In [ ]:
# --- BÀI 3*: RESNET-18 (So sánh Không Augment vs Có Augment) ---
!python train.py --model resnet18 --no_augment --epochs 20
!python evaluate.py --ckpt outputs/resnet18_noaug/best.pt

!python train.py --model resnet18 --epochs 20
!python evaluate.py --ckpt outputs/resnet18_aug/best.pt

In [ ]:
# --- BÀI 4*: PRETRAINED RESNET-50 (Feature Extraction vs Fine-Tuning) ---
!python train.py --model resnet50_fe --epochs 15 --lr 1e-3
!python evaluate.py --ckpt outputs/resnet50_fe/best.pt

!python train.py --model resnet50_ft --epochs 10 --lr 1e-4
!python evaluate.py --ckpt outputs/resnet50_ft/best.pt

## Hiển thị Đồ thị & Bảng Tổng hợp Kết quả

In [ ]:
import pandas as pd
from IPython.display import Image, display
from pathlib import Path

# 1. Hiển thị bảng tổng hợp
summary_path = Path("outputs/summary_table.csv")
if summary_path.is_file():
    df = pd.read_csv(summary_path)
    display(df)

# 2. Hiển thị biểu đồ đối chiếu overfitting ResNet-18
cmp_path = Path("outputs/resnet18_overfitting_comparison.png")
if cmp_path.is_file():
    print("\nBiểu đồ đối chiếu Overfitting của ResNet-18:")
    display(Image(str(cmp_path)))

# 3. Hiển thị ma trận nhầm lẫn của các mô hình tiêu biểu
for m in ["lenet_avg", "googlenet", "resnet18_aug", "resnet50_ft"]:
    p = Path(f"outputs/{m}/confusion_matrix.png")
    if p.is_file():
        print(f"\nMa trận nhầm lẫn - {m}:")
        display(Image(str(p)))